In [ ]:
import os
import glob
import time
import logging
import argparse
from itertools import combinations
from typing import Sequence, Union, Optional


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr, spearmanr
from scipy import stats
import warnings
import traceback
from statsmodels.tsa.stattools import grangercausalitytests

# Import DataSource and Kalman filtering
from src.data.data_loading import DataSource
from src.data.data_preparation import apply_kalman_filter

# Import common preprocessing functions for datasets and scaling
from src.common.batch_preprocessing import (
    load_and_preprocess_data,
    prepare_datasets,
    scale_and_reshape_data,
    prepare_normal_inputs,
    prepare_branch_inputs,
)

# Import evaluation functions
from src.evaluation.evaluation import evaluate_and_plot, display_welch_t_test_results, display_descriptive_statistics, plot_comparison

# Import configuration
from src.common.config_wells import DATA_SOURCES

# Adjust working directory to project root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
print(f"Working directory set to: {os.getcwd()}")

| **Variável**                   | **Descrição** (significado)                                                    | **Unidade de Medida**               | **Fonte da Definição**                                                              |
| ------------------------------ | ------------------------------------------------------------------------------ | ----------------------------------- | ----------------------------------------------------------------------------------- |
| **DATEPRD**                    | Data do registro da produção (data do dia de produção)                         | — (formato de data, ex. yyyy-mm-dd) | Definição implícita no conjunto de dados; fornecido como data (yy.mm.dd).                                   |
| **ON\_STREAM\_HRS**            | Horas em produção no dia (tempo em que o poço esteve fluindo)                  | h (horas)                           | “Operating hours” – horas de operação.                                              |
| **AVG\_DOWNHOLE\_PRESSURE**    | Pressão média de fundo do poço (pressão no fundo durante o dia)                | bar                                 | “Average bottomhole pressure” (pressão média no fundo).                             |
| **AVG\_DOWNHOLE\_TEMPERATURE** | Temperatura média de fundo do poço                                             | °C (graus Celsius)                  | “Average bottomhole temperature” (temperatura média no fundo).                      |
| **AVG\_DP\_TUBING**            | Pressão diferencial média no *tubing* (queda de pressão na coluna de produção) | bar                                 | “Average well differential pressure” – diferença de pressão média no poço (tubing). |
| **AVG\_ANNULUS\_PRESS**        | Pressão média no anular (espaço anular entre colunas)                          | bar                                 | “Average annular pressure” (pressão média no anular).                               |
| **AVG\_CHOKE\_SIZE\_P**        | Abertura média do *choke* (válvula estranguladora) em porcentagem              | %                                   | “Average choke size” – abertura média do choke em %.                                |
| **AVG\_CHOKE\_UOM**            | Unidade de medida da abertura do *choke* (por exemplo, "%")                    | — (texto, ex.: "%")                 | Indica a unidade usada para o choke; no dataset é porcentagem.                      |
| **AVG\_WHP\_P**                | Pressão média na cabeça do poço (Wellhead Pressure)                            | bar                                 | “Average wellhead pressure” (pressão média na cabeça do poço).                      |
| **AVG\_WHT\_P**                | Temperatura média na cabeça do poço (Wellhead Temperature)                     | °C                                  | “Average wellhead temperature” (temperatura média na cabeça do poço).               |
| **DP\_CHOKE\_SIZE**            | Queda de pressão no *choke* (diferença de pressão através da válvula)          | bar                                 | “Differential pressure across the choke” (queda de pressão no choke).               |
| **BORE\_OIL\_VOL**             | Volume de óleo produzido no dia (por poço)                                     | m³/d (metros cúbicos por dia)       | “Volume of oil produced” (medido em m³ por dia no Volve).                           |
| **BORE\_GAS\_VOL**             | Volume de gás produzido no dia (por poço, em condições padrão)                 | m³/d (Sm³/d)                        | “Gas Volume from Well” (volume de gás do poço, m³ diário).                          |
| **BORE\_WAT\_VOL**             | Volume de água produzida no dia (água produzida por poço)                      | m³/d                                | “Water Volume from Well” (volume de água produzido, m³ por dia).                    |
| **BORE\_WI\_VOL**              | Volume de água injetada no dia (água de injeção no poço)                       | m³/d                                | “Water Volume Injected” (volume de água injetada, m³ por dia).                      |
| **FLOW\_KIND**                 | Tipo de fluxo no poço (produção ou injeção)                                    | — (categoria)                       | Indica se o fluxo do dia é de produção ou de injeção.                               |               |

In [ ]:
# =============================================================================
# Statistical Hypothesis Testing Module
# =============================================================================
import numpy as np
import pandas as pd
import scipy.stats as stats
import warnings
from IPython.display import display, HTML

# --- Helper for Displaying Results ---
def display_test_results(results, title):
    """Displays test results in a formatted table."""
    if not results:
        print(f"{title}: No results to display.")
        return

    df_results = pd.DataFrame(results)
    # Basic styling for better readability in notebooks
    styled_df = df_results.style.set_caption(title)\
                              .set_table_styles([{'selector': 'caption',
                                                  'props': [('font-size', '16px'),
                                                            ('font-weight', 'bold'),
                                                            ('text-align', 'center')]}])\
                              .format({"P-value": "{:.4g}", "Statistic": "{:.4f}",
                                       "Lower Bound": "{:.4f}", "Upper Bound": "{:.4f}",
                                       "Outlier %": "{:.2f}%"})
    display(HTML(styled_df.to_html()))


class StatisticalHypothesisTester:
    """
    A module to perform statistical tests, check data quality against physical
    limits, detect outliers, and generate hypotheses about data discrepancies
    between two well datasets.
    """
    def __init__(self, df1, df2, label1, label2, features, physical_limits=None):
        """
        Initializes the tester.

        Args:
            df1 (pd.DataFrame): DataFrame for the first well.
            df2 (pd.DataFrame): DataFrame for the second well.
            label1 (str): Label for the first well.
            label2 (str): Label for the second well.
            features (list): List of features to analyze.
            physical_limits (dict, optional): Dictionary mapping feature names
                                              to tuples (min_val, max_val) for
                                              plausibility checks. Defaults to None.
        """
        self.df1 = df1.copy()
        self.df2 = df2.copy()
        self.label1 = label1
        self.label2 = label2
        self.features = [f for f in features if f in df1.columns and f in df2.columns]
        self.physical_limits = physical_limits if physical_limits else {}
        self.results = {} # To store results from different tests
        self.hypotheses = [] # To store generated hypotheses

    def _check_feature_presence(self, feature):
        """Internal helper to check if feature exists in both dataframes."""
        if feature not in self.df1.columns or feature not in self.df2.columns:
            warnings.warn(f"Feature '{feature}' not found in both datasets. Skipping tests.")
            return False
        if not pd.api.types.is_numeric_dtype(self.df1[feature]) or \
           not pd.api.types.is_numeric_dtype(self.df2[feature]):
             warnings.warn(f"Feature '{feature}' is not numeric in both datasets. Skipping numeric tests.")
             return False
        return True

    def test_distribution_differences_ks(self, alpha=0.05):
        """
        Performs Kolmogorov-Smirnov tests to compare feature distributions.
        Hypothesis: The distributions of the feature are the same between wells.
        """
        print(f"\n--- Kolmogorov-Smirnov (KS) Tests ({self.label1} vs {self.label2}) ---")
        ks_results = []
        for feature in self.features:
            if not self._check_feature_presence(feature): continue

            data1 = self.df1[feature].dropna()
            data2 = self.df2[feature].dropna()

            if len(data1) < 2 or len(data2) < 2:
                result = {"Feature": feature, "Statistic": np.nan, "P-value": np.nan, "Significance": "Insufficient data"}
            else:
                try:
                    stat_val, p_value = stats.ks_2samp(data1, data2)
                    significance = f"Significant Diff. (p<{alpha})" if p_value < alpha else "Not Significant"
                    result = {"Feature": feature, "Statistic": stat_val, "P-value": p_value, "Significance": significance}

                    # Generate Hypothesis based on KS test
                    if p_value < alpha:
                         self.hypotheses.append(
                             f"H_KS_{feature}: The distribution shape of '{feature}' differs significantly "
                             f"between {self.label1} and {self.label2} (KS test p={p_value:.3g}), "
                             f"potentially affecting models sensitive to feature distributions."
                         )

                except Exception as e:
                    result = {"Feature": feature, "Statistic": np.nan, "P-value": np.nan, "Significance": f"Error: {e}"}
            ks_results.append(result)

        self.results['ks_tests'] = ks_results
        display_test_results(ks_results, f"Kolmogorov-Smirnov Tests ({self.label1} vs {self.label2})")
        print("--- End KS Tests ---")

    def check_physical_plausibility(self):
        """
        Checks features against predefined physical limits.
        """
        print("\n--- Physical Plausibility Checks ---")
        plausibility_results = []
        violation_found = False
        for feature, limits in self.physical_limits.items():
            if feature not in self.features:
                warnings.warn(f"Physical limit specified for '{feature}', but it's not in the common features list.")
                continue
            if not self._check_feature_presence(feature): continue

            min_val, max_val = limits
            violations1 = self.df1[feature][(self.df1[feature] < min_val) | (self.df1[feature] > max_val)].count()
            violations2 = self.df2[feature][(self.df2[feature] < min_val) | (self.df2[feature] > max_val)].count()
            percent1 = (violations1 / len(self.df1[feature].dropna())) * 100 if len(self.df1[feature].dropna()) > 0 else 0
            percent2 = (violations2 / len(self.df2[feature].dropna())) * 100 if len(self.df2[feature].dropna()) > 0 else 0

            result = {
                "Feature": feature,
                "Limit Min": min_val,
                "Limit Max": max_val,
                f"{self.label1} Violations (#)": violations1,
                f"{self.label1} Violations (%)": percent1,
                f"{self.label2} Violations (#)": violations2,
                f"{self.label2} Violations (%)": percent2,
            }
            plausibility_results.append(result)

            # Generate Hypothesis based on violations
            if violations1 > 0:
                violation_found = True
                self.hypotheses.append(
                    f"H_PHY_{feature}_{self.label1}: Physically implausible values found for '{feature}' "
                    f"in {self.label1} ({violations1} instances outside [{min_val}, {max_val}]). "
                    f"This could compromise the physics-based layer's reliability for this well."
                )
            if violations2 > 0:
                 violation_found = True
                 self.hypotheses.append(
                    f"H_PHY_{feature}_{self.label2}: Physically implausible values found for '{feature}' "
                    f"in {self.label2} ({violations2} instances outside [{min_val}, {max_val}]). "
                    f"This could compromise the physics-based layer's reliability for this well."
                 )

        self.results['plausibility_checks'] = plausibility_results
        if plausibility_results:
             display_test_results(plausibility_results, "Physical Plausibility Checks")
        else:
             print("No physical limits configured or applicable.")
        if not violation_found:
            print("No physical limit violations detected.")

        print("--- End Plausibility Checks ---")


    def detect_outliers_iqr(self, factor=1.5):
        """
        Detects outliers using the Interquartile Range (IQR) method.
        """
        print(f"\n--- Outlier Detection (IQR Method, factor={factor}) ---")
        outlier_results = []
        outlier_found = False
        for feature in self.features:
            if not self._check_feature_presence(feature): continue

            data1 = self.df1[feature].dropna()
            data2 = self.df2[feature].dropna()

            q1_1, q3_1 = data1.quantile([0.25, 0.75])
            iqr_1 = q3_1 - q1_1
            lower_1, upper_1 = q1_1 - factor * iqr_1, q3_1 + factor * iqr_1
            outliers1 = data1[(data1 < lower_1) | (data1 > upper_1)].count()
            percent1 = (outliers1 / len(data1)) * 100 if len(data1) > 0 else 0


            q1_2, q3_2 = data2.quantile([0.25, 0.75])
            iqr_2 = q3_2 - q1_2
            lower_2, upper_2 = q1_2 - factor * iqr_2, q3_2 + factor * iqr_2
            outliers2 = data2[(data2 < lower_2) | (data2 > upper_2)].count()
            percent2 = (outliers2 / len(data2)) * 100 if len(data2) > 0 else 0


            result = {
                "Feature": feature,
                f"{self.label1} Lower Bound": lower_1,
                f"{self.label1} Upper Bound": upper_1,
                f"{self.label1} Outliers (#)": outliers1,
                f"{self.label1} Outlier %": percent1,
                f"{self.label2} Lower Bound": lower_2,
                f"{self.label2} Upper Bound": upper_2,
                f"{self.label2} Outliers (#)": outliers2,
                f"{self.label2} Outlier %": percent2,
            }
            outlier_results.append(result)

            # Generate Hypothesis based on significant outlier presence (e.g., > 5% or large count diff)
            if outliers1 > 0 or outliers2 > 0:
                outlier_found = True
                if percent1 > 5 or (outliers1 > 10 and percent1 > percent2 * 2): # Example trigger
                     self.hypotheses.append(
                        f"H_OUT_{feature}_{self.label1}: A notable number/percentage of outliers detected for '{feature}' "
                        f"in {self.label1} ({outliers1} instances, {percent1:.1f}%) using IQR. "
                        f"These could disproportionately affect model training or performance."
                    )
                if percent2 > 5 or (outliers2 > 10 and percent2 > percent1 * 2): # Example trigger
                     self.hypotheses.append(
                        f"H_OUT_{feature}_{self.label2}: A notable number/percentage of outliers detected for '{feature}' "
                        f"in {self.label2} ({outliers2} instances, {percent2:.1f}%) using IQR. "
                        f"These could disproportionately affect model training or performance."
                    )


        self.results['outlier_detection_iqr'] = outlier_results
        display_test_results(outlier_results, f"Outlier Detection (IQR, factor={factor})")
        if not outlier_found:
            print("No significant outliers detected via IQR method.")
        print("--- End Outlier Detection ---")

    def run_all_tests(self, run_ks=True, run_plausibility=True, run_outliers=True):
        """Runs all configured tests."""
        print(f"\n{'='*20} Starting Hypothesis Testing {'='*20}")
        if run_ks:
            self.test_distribution_differences_ks()
        if run_plausibility and self.physical_limits:
            self.check_physical_plausibility()
        elif run_plausibility:
            print("\nSkipping physical plausibility check: No 'physical_limits' provided.")
        if run_outliers:
            self.detect_outliers_iqr()

        # Add hypotheses based on existing T-tests (needs t-test results passed or re-run)
        # This part assumes t-test results are available externally for now.
        # We will integrate this better by passing t-test results later.

        print(f"\n{'='*20} Hypothesis Testing Summary {'='*20}")

    def generate_hypotheses_from_ttest(self, ttest_results):
        """Generates hypotheses based on significant t-test results."""
        if not ttest_results:
            return

        print("\n--- Hypotheses Generation from T-Tests ---")
        for result in ttest_results:
             if result.get("Significance") and "Significant" in result["Significance"]:
                 feature = result["Feature"]
                 p_value = result["P-value"]
                 self.hypotheses.append(
                     f"H_TTEST_{feature}: The mean of '{feature}' differs significantly "
                     f"between {self.label1} and {self.label2} (t-test p={p_value:.3g}). "
                     f"This difference in central tendency might impact model performance, "
                     f"especially if '{feature}' is a key predictor."
                 )
        print(f"Added hypotheses based on {len([h for h in self.hypotheses if h.startswith('H_TTEST')])} significant t-tests.")

    def get_summary(self):
        """Returns the collected results and generated hypotheses."""
        print("\n--- Generated Hypotheses ---")
        if not self.hypotheses:
            print("No specific hypotheses were generated based on the performed tests.")
        else:
            for i, hypo in enumerate(self.hypotheses):
                print(f"{i+1}. {hypo}")
        print("--- End of Hypotheses ---")
        # Returning raw results dict for potential further processing if needed
        return {"results": self.results, "hypotheses": self.hypotheses}


# =============================================================================
# Modifications to Existing Code
# =============================================================================

# --- Modify `compare_well_data` ---
# Add `StatisticalHypothesisTester` execution

def compare_well_data(config_1, config_2, common_config, DataSource):
    """
    Orchestrates the loading, analysis, and comparison of data for two wells,
    using well-specific configuration dictionaries. Incorporates hypothesis testing.

    Args:
        config_1 (dict): Configuration for the first well/source.
        config_2 (dict): Configuration for the second well/source.
        common_config (dict): Analysis configuration (e.g., 'core_features',
                              'smoothing_config', 'physical_limits').
        DataSource: External data source class or object used for loading data.
    """
    # ... (Keep existing setup code: well IDs, labels, features, smoothing) ...
    well_id_1 = config_1['wells'][0]
    well_id_2 = config_2['wells'][0]
    label1 = f"{config_1['name']}_{well_id_1.replace('/', '-')}"
    label2 = f"{config_2['name']}_{well_id_2.replace('/', '-')}"
    print(f"Starting data comparison between Well {label1} and Well {label2}")

    core_features = common_config.get('core_features', [])
    if not core_features:
        raise ValueError("common_config must contain a list of 'core_features'.")

    smoothing_config = common_config.get('smoothing_config', {})
    smooth_features = smoothing_config.get('features', [])
    window_size = smoothing_config.get('window_size', 0)
    physical_limits = common_config.get('physical_limits', {}) # <-- Get physical limits

    # --- 1. Load Data ---
    # ... (Keep existing data loading) ...
    print(f"Loading data for {label1}...")
    # Assuming load_and_preprocess_data exists and works
    df_raw1 = load_and_preprocess_data(DataSource, config_1, core_features, well_id_1)
    print(f"Loading data for {label2}...")
    df_raw2 = load_and_preprocess_data(DataSource, config_2, core_features, well_id_2)
    dfs_raw = {label1: df_raw1, label2: df_raw2}


    # --- 2. Apply Smoothing (Optional) ---
    # ... (Keep existing smoothing logic) ...
    dfs_smooth = {}
    if smooth_features and window_size > 1:
        print(f"\nApplying smoothing (window={window_size}) to {smooth_features}...")
        dfs_smooth[label1] = apply_rolling_smoothing(df_raw1, smooth_features, window_size)
        dfs_smooth[label2] = apply_rolling_smoothing(df_raw2, smooth_features, window_size)
        active_dfs = dfs_smooth
        active_features = smooth_features
        active_label_suffix = "_Smooth"
        print("Using SMOOTHED data for subsequent analysis where applicable.")
    else:
        print("\nSkipping smoothing step. Using RAW data for analysis.")
        active_dfs = dfs_raw
        active_features = core_features # Analyze all core features if not smoothing
        active_label_suffix = ""


    # --- 3. Statistical Analysis (Existing) ---
    print("\n--- Descriptive Statistics (Raw Data) ---")
    combined_stats_raw = get_combined_statistics(dfs_raw, core_features)
    display_descriptive_statistics(combined_stats_raw, title="Descriptive Statistics (Raw Data)") # Assume display func exists
    # Perform t-tests on RAW data always for baseline mean comparison
    t_test_results_raw = perform_t_tests(df_raw1, df_raw2, core_features, label1, label2)

    # Perform t-tests on the 'active' data (raw or smoothed) if different from raw
    t_test_results_active = t_test_results_raw
    if active_dfs is not dfs_raw:
        print("\n--- Descriptive Statistics (Smoothed Data) ---")
        combined_stats_smooth = get_combined_statistics(active_dfs, active_features)
        display_descriptive_statistics(combined_stats_smooth, title=f"Descriptive Statistics ({active_label_suffix.strip('_')} Data)")
        t_test_results_active = perform_t_tests(active_dfs[label1], active_dfs[label2], active_features,
                                               f"{label1}{active_label_suffix}", f"{label2}{active_label_suffix}")


    # --- 3b. NEW: Hypothesis Testing Module ---
    print("\n\n" + "="*30 + f" Running Statistical Hypothesis Tester on {'Smoothed' if active_dfs is dfs_smooth else 'Raw'} Data " + "="*30)
    # Use the 'active' dataframes (smoothed if available, else raw) for hypothesis testing
    tester = StatisticalHypothesisTester(
        active_dfs[label1], active_dfs[label2],
        f"{label1}{active_label_suffix}", f"{label2}{active_label_suffix}",
        active_features, # Test on the features that were smoothed, or all core if no smoothing
        physical_limits
    )
    # Generate hypotheses based on the relevant t-tests (raw or smoothed)
    tester.generate_hypotheses_from_ttest(t_test_results_active)
    # Run the other tests
    tester.run_all_tests()
    # Get the summary
    summary = tester.get_summary()
    print("="*80)


    # --- 4. Visualizations (Existing) ---
    # Adjust features for plots based on whether smoothing was done
    features_for_plots = active_features if active_dfs is dfs_smooth else core_features
    plot_label_prefix = f"{active_label_suffix.strip('_')} Data" if active_label_suffix else "Raw Data"

    print(f"\n--- Generating Comparative Time Series Plots ({plot_label_prefix}) ---")
    plot_comparative_time_series_all(active_dfs, features_for_plots, set_name_prefix=plot_label_prefix)
    print(f"\n--- Generating Comparative Distribution Plots ({plot_label_prefix}) ---")
    plot_comparative_distribution_all(active_dfs, features_for_plots, set_name_prefix=plot_label_prefix)

    # Correlation matrices are often best on raw data to see underlying relationships
    print(f"\n--- Generating Correlation Matrix for {label1} (Raw Data) ---")
    plot_correlation_matrix(df_raw1, core_features, ds=config_1, method="pearson")
    print(f"\n--- Generating Correlation Matrix for {label2} (Raw Data) ---")
    plot_correlation_matrix(df_raw2, core_features, ds=config_2, method="pearson")

    # Optional: Show correlations on smoothed data if smoothing was applied
    if active_dfs is dfs_smooth:
        print(f"\n--- Generating Correlation Matrix for {label1} (Smoothed Data) ---")
        plot_correlation_matrix(active_dfs[label1], active_features, ds=config_1, method="pearson")
        print(f"\n--- Generating Correlation Matrix for {label2} (Smoothed Data) ---")
        plot_correlation_matrix(active_dfs[label2], active_features, ds=config_2, method="pearson")


    print(f"\nComparison and Hypothesis Testing between Well {label1} and Well {label2} finished.")
    print("Review the generated hypotheses above for potential explanations of performance differences.")
    
    
# =============================================================================
# Additional Hypothesis Testing and Diagnostic Functions
# =============================================================================

import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import plotly.graph_objects as go

def compare_feature_distributions_ks(df1, df2, feature, label1, label2):
    """
    Compares the distribution of a specific feature between two wells using the
    Kolmogorov–Smirnov (KS) test.
    
    Args:
        df1 (pd.DataFrame): DataFrame for well label1.
        df2 (pd.DataFrame): DataFrame for well label2.
        feature (str): Feature name to compare.
        label1 (str): Label for the first well.
        label2 (str): Label for the second well.
    
    Returns:
        dict: Dictionary containing KS statistic, P-value, and significance flag.
    """
    if feature in df1.columns and feature in df2.columns:
        data1 = df1[feature].dropna()
        data2 = df2[feature].dropna()
        if len(data1) > 0 and len(data2) > 0:
            ks_stat, ks_p = stats.ks_2samp(data1, data2)
            print(f"\n--- KS Test for {feature} ({label1} vs {label2}) ---")
            print(f"KS Statistic: {ks_stat:.3f}, P-value: {ks_p:.3f}")
            return {"Feature": feature, "KS Statistic": ks_stat, "P-value": ks_p, 
                    "Significance": "Significant Diff." if ks_p < 0.05 else ""}
    print(f"Feature '{feature}' missing in one of the datasets for KS test.")
    return {}

def check_outliers_against_physical_limits(df, feature, min_val, max_val):
    """
    Checks for out-of-bound values for a given feature based on specified physical limits.
    
    Args:
        df (pd.DataFrame): DataFrame containing well data.
        feature (str): Feature name to check.
        min_val (float): Minimum acceptable value.
        max_val (float): Maximum acceptable value.
    
    Returns:
        pd.DataFrame: A DataFrame containing rows with outlier values.
    """
    if feature in df.columns:
        outliers = df[(df[feature] < min_val) | (df[feature] > max_val)]
        print(f"\nFeature '{feature}' - Outliers (Values outside {min_val} and {max_val}): {len(outliers)} found.")
        if not outliers.empty:
            print(outliers[[feature]])
        else:
            print("No outliers detected.")
        return outliers
    else:
        print(f"Feature '{feature}' not found in the provided DataFrame.")
        return pd.DataFrame()

def generate_hypotheses_from_stats_summary(summary_df, threshold=0.1):
    """
    Generates hypotheses based on differences in means across wells.
    
    Args:
        summary_df (pd.DataFrame): Combined descriptive statistics from multiple wells.
        threshold (float): Relative difference threshold (e.g., 10%) to trigger a hypothesis.
    
    Returns:
        list: A list of hypothesis strings.
    """
    hypotheses = []
    # summary_df is expected to have a multi-index where the inner level contains 'mean'
    if summary_df.empty:
        print("Summary DataFrame is empty. No hypotheses generated.")
        return hypotheses

    for feature in summary_df.index:
        if summary_df.columns.nlevels >= 2:
            try:
                # Extract mean values from each well's statistics
                means = summary_df.loc[feature].xs('mean', level=1, axis=1)
            except Exception as e:
                warnings.warn(f"Could not extract mean for {feature}: {e}")
                continue
            if len(means) >= 2:
                # Compute relative difference between means from two wells
                mean_diff = abs(means.iloc[0] - means.iloc[1])
                avg_mean = means.mean()
                rel_diff = mean_diff / avg_mean if avg_mean != 0 else 0
                if rel_diff > threshold:
                    hypothesis = (f"Hypothesis: The feature '{feature}' has a relative mean difference of "
                                  f"{rel_diff:.2f} between wells, which may contribute to performance differences.")
                    print(hypothesis)
                    hypotheses.append(hypothesis)
    if not hypotheses:
        print("No significant mean differences detected based on the threshold.")
    return hypotheses

def run_statistical_tests_on_features(df_list, features, labels):
    """
    Runs both the Welch's t-test and the Kolmogorov–Smirnov (KS) test on specified features
    across two wells.
    
    Args:
        df_list (list): List containing exactly two DataFrames.
        features (list): List of feature names on which to perform tests.
        labels (list): List of well labels corresponding to the DataFrames.
    
    Returns:
        dict: A dictionary summarizing the results of the t-test and KS test for each feature.
    """
    results = {}
    if len(df_list) != 2 or len(labels) != 2:
        print("This function is designed for comparing exactly two wells.")
        return results

    df1, df2 = df_list
    label1, label2 = labels

    # Existing t-test function is reused
    results['t_test'] = perform_t_tests(df1, df2, features, label1, label2)
    
    ks_results = []
    for feature in features:
        ks_result = compare_feature_distributions_ks(df1, df2, feature, label1, label2)
        if ks_result:
            ks_results.append(ks_result)
    results['ks_test'] = ks_results
    return results

def visualize_feature_behavior_over_time(well_data, feature):
    """
    Visualizes the temporal behavior of a specified feature from well data using matplotlib.
    
    Args:
        well_data (pd.DataFrame): DataFrame containing time-series data (indexed by time).
        feature (str): Feature name to visualize.
    
    Returns:
        None: Displays a matplotlib plot.
    """
    if feature not in well_data.columns:
        print(f"Feature '{feature}' not found in the well data.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(well_data.index, well_data[feature], label=feature)
    plt.title(f"Time Series of {feature}")
    plt.xlabel("Time")
    plt.ylabel(feature)
    plt.legend()
    plt.grid(True)
    plt.show()


# =============================================================================
# Extended Orchestration Function
# =============================================================================

def compare_well_data_extended(config_1, config_2, common_config, DataSource):
    """
    Extended version of compare_well_data that includes additional hypothesis testing routines.
    
    In addition to the original comparison, the function:
      - Runs KS tests to compare feature distributions.
      - Checks for out-of-bound values for physically critical features.
      - Generates hypotheses based on differences in the descriptive statistics.
    
    Args:
        config_1 (dict): Configuration for the first well/source.
        config_2 (dict): Configuration for the second well/source.
        common_config (dict): Analysis configuration (e.g., 'core_features', 'smoothing_config').
        DataSource: External data source used for loading data.
    """
    # Run the existing well data comparison workflow.
    compare_well_data(config_1, config_2, common_config, DataSource)

    # Post-analysis extension: Additional hypothesis tests.
    well_id_1 = config_1['wells'][0]
    well_id_2 = config_2['wells'][0]
    label1 = f"{config_1['name']}_{well_id_1.replace('/', '-')}"
    label2 = f"{config_2['name']}_{well_id_2.replace('/', '-')}"
    
    core_features = common_config.get('core_features', [])
    smoothing_config = common_config.get('smoothing_config', {})
    smooth_features = smoothing_config.get('features', [])
    
    # Load raw data (reloading if needed)
    df_raw1 = load_and_preprocess_data(DataSource, config_1, core_features, well_id_1)
    df_raw2 = load_and_preprocess_data(DataSource, config_2, core_features, well_id_2)
    
    # Run additional statistical tests (t-test and KS test) on raw data.
    print("\n--- Running Additional Statistical Tests on Raw Data ---")
    additional_results = run_statistical_tests_on_features([df_raw1, df_raw2], core_features, [label1, label2])
    
    # Example: Check physical limits for key features.
    # (Adjust the physical limits based on domain knowledge.)
    print("\n--- Checking Outliers Against Physical Limits ---")
    physical_limits = {
        'PI': (0, np.inf),  # PI should be strictly positive
        'AVG_DOWNHOLE_PRESSURE': (500, 5000),  # Example limits (e.g., in psi)
        'AVG_WHP_P': (50, 5000)  # Example limits (e.g., in psi)
    }
    for feature, (min_val, max_val) in physical_limits.items():
        print(f"\nChecking Well {label1} for feature '{feature}'")
        check_outliers_against_physical_limits(df_raw1, feature, min_val, max_val)
        print(f"Checking Well {label2} for feature '{feature}'")
        check_outliers_against_physical_limits(df_raw2, feature, min_val, max_val)
    
    # Generate hypotheses from descriptive statistics.
    print("\n--- Generating Hypotheses from Descriptive Statistics ---")
    combined_stats_raw = get_combined_statistics({label1: df_raw1, label2: df_raw2}, core_features)
    hypotheses = generate_hypotheses_from_stats_summary(combined_stats_raw)
    if hypotheses:
        print("\nGenerated Hypotheses:")
        for hypo in hypotheses:
            print(f"- {hypo}")
    else:
        print("No strong hypotheses generated based on current descriptive statistics.")

In [ ]:
# =============================================================================
# Reusable Utility Functions
# =============================================================================

def apply_rolling_smoothing(df, features_to_smooth, window_size):
    """
    Applies rolling mean smoothing to specified features in a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.
        features_to_smooth (list): List of column names to smooth.
        window_size (int): The rolling window size.

    Returns:
        pd.DataFrame: DataFrame with smoothed features.
    """
    df_smooth = df.copy()
    if not features_to_smooth or window_size <= 1:
        print("Skipping smoothing (no features specified or window size <= 1).")
        return df_smooth

    print(f"Applying rolling window (size={window_size}) to: {', '.join(features_to_smooth)}")
    for feature in features_to_smooth:
        if feature in df_smooth.columns:
            df_smooth[feature] = df_smooth[feature].rolling(window=window_size, min_periods=1).mean()
        else:
            warnings.warn(f"Feature '{feature}' not found in DataFrame for smoothing. Skipping.")
    return df_smooth


def get_combined_statistics(df_dict, features):
    """
    Calculates and combines descriptive statistics for multiple DataFrames.

    Args:
        df_dict (dict): Dictionary where keys are labels (e.g., well identifiers)
                        and values are DataFrames.
        features (list): List of features to calculate statistics for.

    Returns:
        pd.DataFrame: Combined descriptive statistics.
    """
    stats_list = []
    labels = []
    for label, df in df_dict.items():
        valid_features = [f for f in features if f in df.columns]
        if not valid_features:
            warnings.warn(f"No specified features found in DataFrame '{label}'. Skipping statistics.")
            continue
        stats_list.append(df[valid_features].describe().T)
        labels.append(label)

    if not stats_list:
        return pd.DataFrame()  # Return empty DataFrame if no stats were computed

    return pd.concat(stats_list, axis=1, keys=labels)


def perform_t_tests(df1, df2, features, label1, label2):
    print(f"\n--- Welch's T-Tests ({label1} vs {label2}) ---")
    print("(Note: Assumes independence, interpret p-values cautiously for time series)")
    results = []  # Lista para armazenar os resultados
    for feature in features:
        if feature in df1.columns and feature in df2.columns:
            if pd.api.types.is_numeric_dtype(df1[feature]) and pd.api.types.is_numeric_dtype(df2[feature]):
                data1 = df1[feature].dropna()
                data2 = df2[feature].dropna()
                if len(data1) > 1 and len(data2) > 1:
                    try:
                        stat_val, p_value = stats.ttest_ind(data1, data2, equal_var=False)
                        significance = "Significant Diff." if p_value < 0.05 else ""
                        results.append({
                            "Feature": feature,
                            "Statistic": stat_val,
                            "P-value": p_value,
                            "Significance": significance
                        })
                    except Exception as e:
                        results.append({
                            "Feature": feature,
                            "Statistic": float('nan'),
                            "P-value": float('nan'),
                            "Significance": f"Error: {e}"
                        })
                else:
                    results.append({
                        "Feature": feature,
                        "Statistic": float('nan'),
                        "P-value": float('nan'),
                        "Significance": "Insufficient data"
                    })
    print("--- End T-Tests ---")
    display_welch_t_test_results(results, title=f"Welch's T-Tests ({label1} vs {label2})")
    return results

# =============================================================================
# New Plotting Functions
# =============================================================================

def plot_comparative_time_series_individual(df_dict, feature, set_name="Time Series Comparison"):
    """
    Generates an independent comparative time series plot for a given feature.
    Extracts the series from exactly two DataFrames and calls the new plot_comparison function.
    
    Args:
        df_dict (dict): Dictionary with exactly two entries; keys are well labels and values are DataFrames.
        feature (str): Feature name to plot.
        set_name (str): Title prefix (unused in this focused version).
    """
    keys = list(df_dict.keys())
    if len(keys) != 2:
        print("Comparative time series plot requires exactly two DataFrames.")
        return

    label1, label2 = keys
    series1 = df_dict[label1][feature].dropna().values
    series2 = df_dict[label2][feature].dropna().values
    if min(len(series1), len(series2)) == 0:
        print(f"One of the datasets has no data for feature '{feature}'. Skipping plot.")
        return

    well = f"{label1} vs {label2}"
    plot_comparison(series1, series2, feature, well)
    
def plot_comparative_distribution_individual(df_dict, feature, set_name="Distribution Comparison"):
    """
    Generates an independent overlaid histogram to compare the distribution of a given feature
    between two wells using a Plotly design.

    Args:
        df_dict (dict): Dictionary with exactly two entries; keys are well labels and values are DataFrames.
        feature (str): Feature name to plot.
        set_name (str): Title prefix used for the plot.
    """
    keys = list(df_dict.keys())
    if len(keys) != 2:
        print("Comparative distribution plot requires exactly two DataFrames.")
        return

    label1, label2 = keys
    series1 = df_dict[label1][feature].dropna().values
    series2 = df_dict[label2][feature].dropna().values

    if len(series1) == 0 or len(series2) == 0:
        print(f"Insufficient data for feature '{feature}' in one of the datasets. Skipping plot.")
        return

    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=series1, 
        name=label1, 
        opacity=0.6, 
        marker=dict(line=dict(width=1))
    ))
    fig.add_trace(go.Histogram(
        x=series2, 
        name=label2, 
        opacity=0.6, 
        marker=dict(line=dict(width=1))
    ))
    fig.update_layout(
        barmode='overlay',
        title=dict(text=f"{set_name}: {feature}", x=0.5, font=dict(size=36, color='#2E2E2E')),
        xaxis=dict(title=feature, title_font=dict(size=30, color='#2E2E2E'),
                   tickfont=dict(size=26, color='#2E2E2E')),
        yaxis=dict(title="Frequency", title_font=dict(size=30, color='#2E2E2E'),
                   tickfont=dict(size=26, color='#2E2E2E')),
        legend=dict(orientation="h", x=0.5, y=1.1, xanchor="center", font=dict(size=28)),
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='white',
        width=1200,
        height=600
    )
    fig.show()


def plot_comparative_time_series_all(df_dict, features_to_plot, set_name_prefix="Time Series Comparison"):
    """
    For each specified feature, generates an independent comparative time series chart.
    
    Args:
        df_dict (dict): Dictionary with two keys (well labels) and respective DataFrames.
        features_to_plot (list): List of feature names to plot.
        set_name_prefix (str): Title prefix for the plots.
    """
    for feature in features_to_plot:
        if all(feature in df.columns for df in df_dict.values()):
            plot_comparative_time_series_individual(df_dict, feature, set_name=f"{set_name_prefix}: {feature}")
        else:
            print(f"Feature '{feature}' missing in one of the datasets; skipping time series plot.")


def plot_comparative_distribution_all(df_dict, features_to_plot, set_name_prefix="Distribution Comparison"):
    """
    For each specified feature, generates an independent comparative distribution chart.
    
    Args:
        df_dict (dict): Dictionary with two keys (well labels) and respective DataFrames.
        features_to_plot (list): List of feature names to plot.
        set_name_prefix (str): Title prefix for the plots.
    """
    for feature in features_to_plot:
        if all(feature in df.columns for df in df_dict.values()):
            plot_comparative_distribution_individual(df_dict, feature, set_name=f"{set_name_prefix}: {feature}")
        else:
            print(f"Feature '{feature}' missing in one of the datasets; skipping distribution plot.")

# =============================================================================
# Plot Correlation Matrix Function (unchanged)
# =============================================================================

def plot_correlation_matrix(df, features, ds, method="pearson", color_palette=None, show_feature_mapping=True):
    """
    Create an interactive correlation heatmap using Plotly and optionally display a feature mapping table.

    Args:
        df (pd.DataFrame): Input DataFrame.
        features (list): List of features to compute correlations for.
        ds (dict): Dataset configuration dictionary containing at least the key "name".
        method (str): Correlation method (default is "pearson").
        color_palette (list): Optional list of colors for the correlation heatmap.
        show_feature_mapping (bool): Whether to display a feature mapping table.
    """
    default_colors = ['#206A92', '#FF5733', '#8E44AD', '#E67E22', '#2ECC71']
    colorscale = color_palette if color_palette else default_colors

    corr = df[features].corr(method=method)
    heatmap_fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=features,
        y=features,
        colorscale=colorscale,
        zmin=-1,
        zmax=1,
        text=np.round(corr.values, 2),
        hovertemplate=(
            "Feature X: %{x}<br>" +
            "Feature Y: %{y}<br>" +
            "Correlation: %{z:.2f}<extra></extra>"
        ),
        colorbar=dict(title="Correlation", titleside="right")
    ))

    threshold = 0.75
    annotations = []
    for i, feature_y in enumerate(features):
        for j, feature_x in enumerate(features):
            value = corr.iloc[i, j]
            if i != j and abs(value) >= threshold:
                annotations.append(dict(
                    x=feature_x,
                    y=feature_y,
                    text=f"{value:.2f}",
                    showarrow=False,
                    font=dict(color="black", size=16, family="Arial Black"),
                    bgcolor="rgba(255,255,255,0.7)"
                ))
    heatmap_fig.update_layout(
        title=dict(text=f"{method.capitalize()} Correlation Matrix", x=0.5, font=dict(size=28)),
        xaxis=dict(tickangle=-45, tickfont=dict(size=14)),
        yaxis=dict(tickfont=dict(size=14)),
        annotations=annotations,
        width=1100,
        height=900,
        paper_bgcolor='white',
        plot_bgcolor='rgba(0,0,0,0)'
    )
    heatmap_fig.show()

    if show_feature_mapping:
        if ds["name"].upper() == "VOLVE":
            default_feature_mapping = {
                "BORE_GAS_VOL": "Volume of gas produced (SCF or m³).",
                "CE": "Energy Coefficient, a metric of production efficiency.",
                "AVG_DOWNHOLE_PRESSURE": "Average pressure at the well bottom (psi or bar).",
                "delta_P": "Pressure variation indicating operational changes.",
                "Tempo_Inicio_Prod": "Time since production start (hours/days).",
                "PI": "Well Productivity Index.",
                "AVG_WHP_P": "Average Wellhead Pressure.",
                "BORE_WAT_VOL": "Volume of water produced.",
                "ON_STREAM_HRS": "Hours the well has been in production.",
                "Taxa_Declinio": "Rate of production decline."
            }
        elif ds["name"].upper() == "UNISIM":
            default_feature_mapping = {
                "PWFO": "Well pressure",
                "QOOB": "Oil flow",
                "QWOB": "Water flow",
                "QLOB": "Liquid flow (oil + water)",
                "QGOB": "Gas flow",
            }
        mapping_features = [feat for feat in features if feat in default_feature_mapping]
        mapping_descriptions = [default_feature_mapping.get(feat, "No description available.") for feat in mapping_features]
        table_fig = go.Figure(data=[go.Table(
            columnwidth=[0.25, 0.75],
            header=dict(
                values=["<b>Feature</b>", "<b>Description</b>"],
                fill_color=default_colors[0],
                font=dict(size=18, color='white'),
                align='left'
            ),
            cells=dict(
                values=[mapping_features, mapping_descriptions],
                fill_color=[['lavender'] * len(mapping_features),
                            ['#FFDDC1'] * len(mapping_descriptions)],
                align='left',
                font=dict(size=16, color=['black']),
                height=30
            )
        )])
        table_fig.update_layout(
            title=dict(text="Feature Mapping", x=0.5, font=dict(size=24)),
            width=1100,
            height=700,
            margin=dict(l=10, r=10, t=40, b=10),
            paper_bgcolor='white'
        )
        table_fig.show()


# =============================================================================
# Main Execution Function
# =============================================================================

def main():
    """Main entry point for the well data analysis."""
    pd.set_option('display.max_columns', 50)
    pd.set_option('display.width', 1000)

    # --- Configuration Setup ---
    config_f12 = {
        'name': 'VOLVE',
        'wells': ['15/9-F-12'],
        'load_params': {
            'data_path': 'data/volve/Volve_Equinor.csv',
            'serie_name': 'BORE_OIL_VOL',
            'add_physical_features': False
        },
        'model_path': 'VOLVE_MODELS/best_disruptive_model_VOLVE.keras',
        'target_column': 'BORE_OIL_VOL',
        'variable_mapping': None,
        'features': [
            'BORE_OIL_VOL', 'CE', 'delta_P', 'PI', 'AVG_DOWNHOLE_PRESSURE', 'AVG_WHP_P',
            'BORE_WAT_VOL', 'ON_STREAM_HRS', 'Tempo_Inicio_Prod', 'Taxa_Declinio',
            'BORE_GAS_VOL'
        ]
    }

    config_f14 = {
        'name': 'VOLVE',
        'wells': ['15/9-F-14'],
        'load_params': {
            'data_path': 'data/volve/Volve_Equinor.csv',
            'serie_name': 'BORE_OIL_VOL',
            'add_physical_features': False
        },
        'model_path': 'VOLVE_MODELS/best_disruptive_model_VOLVE.keras',
        'target_column': 'BORE_OIL_VOL',
        'variable_mapping': None,
        'features': [
            'BORE_OIL_VOL', 'CE', 'delta_P', 'PI', 'AVG_DOWNHOLE_PRESSURE', 'AVG_WHP_P',
            'BORE_WAT_VOL', 'ON_STREAM_HRS', 'Tempo_Inicio_Prod', 'Taxa_Declinio',
            'BORE_GAS_VOL'
        ]
    }

    common_analysis_config = {
        'core_features': [
            'BORE_OIL_VOL', 'CE', 'delta_P', 'PI', 'AVG_DOWNHOLE_PRESSURE', 'AVG_WHP_P',
            'BORE_WAT_VOL', 'ON_STREAM_HRS', 'Tempo_Inicio_Prod', 'Taxa_Declinio',
            'BORE_GAS_VOL'
        ],
        'smoothing_config': {
            'features': ['AVG_DOWNHOLE_PRESSURE', 'PI', 'AVG_WHP_P', 'BORE_OIL_VOL'],
            'window_size': 15
        }
    }


    try:
        compare_well_data(config_f12, config_f14, common_analysis_config, DataSource)
        # compare_well_data_extended(config_f12, config_f14, common_analysis_config, DataSource)
    except FileNotFoundError:
        print("Error: Data file not found. Please check the path in the configuration.")
    except ImportError as e:
        print(f"Error: Failed to import a required module: {e}. Check paths and installations.")
    except KeyError as e:
        print(f"Error: Missing key in configuration: {e}. Please check configuration dictionaries.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        traceback.print_exc()


if __name__ == "__main__":
    main()

In [ ]:
def interpret_correlation(corr, p_val):
    """
    Dummy interpretation function.
    Replace with your actual interpretation logic.
    """
    if p_val < 0.05:
        if corr > 0:
            return "Strong positive relationship"
        else:
            return "Strong negative relationship"
    return "Weak or no significant relationship"


def domain_specific_tests_volve(df, main_feature, color_palette=None):
    """
    Domain-specific tests for VOLVE data.
    
    Tests correlations among selected pressure and production features.
    """
    default_colors = ['#206A92', '#FF5733', '#8E44AD', '#E67E22', '#2ECC71']
    palette = color_palette if color_palette else default_colors
    results = []
    
    # Domain groups for VOLVE
    pressure_features = ['AVG_DOWNHOLE_PRESSURE', 'AVG_WHP_P', 'delta_P']
    production_features = [main_feature, 'BORE_GAS_VOL']
    
    # Test correlations between pressure features and production metrics.
    for p in pressure_features:
        if p not in df.columns:
            continue
        for prod in production_features:
            if prod not in df.columns:
                continue
            common_idx = df[[p, prod]].dropna().index
            if len(common_idx) < 2:
                continue
            corr, p_val = pearsonr(df.loc[common_idx, p], df.loc[common_idx, prod])
            interpretation = interpret_correlation(corr, p_val)
            results.append({
                "Feature Pair": f"{p} vs {prod}",
                "Pearson r": f"{corr:.3f}",
                "p-value": f"{p_val:.3e}",
                "Interpretation": interpretation
            })
    
    # Test relationship between PI and pressures (if available)
    if 'PI' in df.columns:
        for p in pressure_features:
            if p not in df.columns:
                continue
            common_idx = df[[p, 'PI']].dropna().index
            if len(common_idx) < 2:
                continue
            corr, p_val = pearsonr(df.loc[common_idx, p], df.loc[common_idx, 'PI'])
            interpretation = interpret_correlation(corr, p_val)
            results.append({
                "Feature Pair": f"{p} vs PI",
                "Pearson r": f"{corr:.3f}",
                "p-value": f"{p_val:.3e}",
                "Interpretation": interpretation
            })
    
    # Test relationship between production start time and the main production metric.
    if 'Tempo_Inicio_Prod' in df.columns:
        common_idx = df[['Tempo_Inicio_Prod', main_feature]].dropna().index
        if len(common_idx) >= 2:
            corr, p_val = pearsonr(df.loc[common_idx, 'Tempo_Inicio_Prod'], df.loc[common_idx, main_feature])
            interpretation = interpret_correlation(corr, p_val)
            results.append({
                "Feature Pair": f"Tempo_Inicio_Prod vs {main_feature}",
                "Pearson r": f"{corr:.3f}",
                "p-value": f"{p_val:.3e}",
                "Interpretation": interpretation
            })
    
    # --- Display results as a Plotly table ---
    feature_pairs = [r["Feature Pair"] for r in results]
    pearson_rs     = [r["Pearson r"]   for r in results]
    p_values       = [r["p-value"]     for r in results]
    interpretations= [r["Interpretation"] for r in results]
    
    table_fig = go.Figure(data=[go.Table(
        columnwidth=[0.4, 0.2, 0.2, 0.4],
        header=dict(
            values=["<b>Feature Pair</b>", "<b>Pearson r</b>", "<b>p-value</b>", "<b>Interpretation</b>"],
            fill_color=palette[0],
            font=dict(color="white", size=16),
            align="center"
        ),
        cells=dict(
            values=[feature_pairs, pearson_rs, p_values, interpretations],
            fill_color="lavender",
            font=dict(color="black", size=14),
            align="center",
            height=40
        )
    )])
    
    table_fig.update_layout(
        title=dict(text="Domain-Specific Correlation Tests (VOLVE)", x=0.5, font=dict(size=24)),
        width=1100,
        height=800,
        paper_bgcolor="white"
    )
    
    table_fig.show()
    return results


def domain_specific_tests_unisim(df, main_feature, color_palette=None):
    """
    Domain-specific tests for UNISIM data.
    
    For UNISIM, we assume:
      - The main production metric is typically the Oil flow (e.g., 'QOOB').
      - Pressure is represented by 'PWFO'.
      - Additional flow variables include: 'QLOB', 'QWOB', 'QGOB'.
      - 'Tempo_Inicio_Prod' (if available) is used to analyze production start time.
    """
    default_colors = ['#206A92', '#FF5733', '#8E44AD', '#E67E22', '#2ECC71']
    palette = color_palette if color_palette else default_colors
    results = []
    
    pressure_feature = 'PWFO'
    additional_flows = ['QLOB', 'QWOB', 'QGOB']
    
    # Test correlation between pressure and the main production metric.
    if pressure_feature in df.columns and main_feature in df.columns:
        common_idx = df[[pressure_feature, main_feature]].dropna().index
        if len(common_idx) >= 2:
            corr, p_val = pearsonr(df.loc[common_idx, pressure_feature], df.loc[common_idx, main_feature])
            interpretation = interpret_correlation(corr, p_val)
            results.append({
                "Feature Pair": f"{pressure_feature} vs {main_feature}",
                "Pearson r": f"{corr:.3f}",
                "p-value": f"{p_val:.3e}",
                "Interpretation": interpretation
            })
    
    # Test correlations between pressure and each additional flow metric.
    for flow in additional_flows:
        if pressure_feature in df.columns and flow in df.columns:
            common_idx = df[[pressure_feature, flow]].dropna().index
            if len(common_idx) < 2:
                continue
            corr, p_val = pearsonr(df.loc[common_idx, pressure_feature], df.loc[common_idx, flow])
            interpretation = interpret_correlation(corr, p_val)
            results.append({
                "Feature Pair": f"{pressure_feature} vs {flow}",
                "Pearson r": f"{corr:.3f}",
                "p-value": f"{p_val:.3e}",
                "Interpretation": interpretation
            })
    
    # Test relationship between production start time and the main production metric.
    if 'Tempo_Inicio_Prod' in df.columns and main_feature in df.columns:
        common_idx = df[['Tempo_Inicio_Prod', main_feature]].dropna().index
        if len(common_idx) >= 2:
            corr, p_val = pearsonr(df.loc[common_idx, 'Tempo_Inicio_Prod'], df.loc[common_idx, main_feature])
            interpretation = interpret_correlation(corr, p_val)
            results.append({
                "Feature Pair": f"Tempo_Inicio_Prod vs {main_feature}",
                "Pearson r": f"{corr:.3f}",
                "p-value": f"{p_val:.3e}",
                "Interpretation": interpretation
            })
    
    # --- Display results as a Plotly table ---
    feature_pairs = [r["Feature Pair"] for r in results]
    pearson_rs     = [r["Pearson r"]   for r in results]
    p_values       = [r["p-value"]     for r in results]
    interpretations= [r["Interpretation"] for r in results]
    
    table_fig = go.Figure(data=[go.Table(
        columnwidth=[0.4, 0.2, 0.2, 0.4],
        header=dict(
            values=["<b>Feature Pair</b>", "<b>Pearson r</b>", "<b>p-value</b>", "<b>Interpretation</b>"],
            fill_color=palette[0],
            font=dict(color="white", size=16),
            align="center"
        ),
        cells=dict(
            values=[feature_pairs, pearson_rs, p_values, interpretations],
            fill_color="lavender",
            font=dict(color="black", size=14),
            align="center",
            height=40
        )
    )])
    
    table_fig.update_layout(
        title=dict(text="Domain-Specific Correlation Tests (UNISIM)", x=0.5, font=dict(size=24)),
        width=1100,
        height=800,
        paper_bgcolor="white"
    )
    
    table_fig.show()
    return results

# =============================================================================
# Helper Functions for Data Source Handling
# =============================================================================

def get_features_and_main(ds, default_selected_features):
    """
    Return the list of features and the main production metric for a given data source.
    
    For VOLVE, use the fixed default list and main feature.
    For UNISIM, use the features provided in ds and ds["target_column"].
    """
    if ds["name"].upper() == "VOLVE":
        features = default_selected_features
        main_feature = "BORE_OIL_VOL"
    elif ds["name"].upper() == "UNISIM":
        features = ds["features"]
        main_feature = ds["target_column"]
    else:
        # Fallback to default settings.
        features = default_selected_features
        main_feature = default_selected_features[0]  # or another sensible default
    return features, main_feature

# =============================================================================
# Your Visualization Functions (Examples)
# =============================================================================

def plot_correlation_matrix(df, features, ds, method="pearson", color_palette=None, show_feature_mapping=True):
    """
    Create an interactive correlation heatmap using Plotly and optionally display a feature mapping table.
    """
    default_colors = ['#206A92', '#FF5733', '#8E44AD', '#E67E22', '#2ECC71']
    colorscale = color_palette if color_palette else default_colors

    corr = df[features].corr(method=method)
    heatmap_fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=features,
        y=features,
        colorscale=colorscale,
        zmin=-1,
        zmax=1,
        text=np.round(corr.values, 2),
        hovertemplate=(
            "Feature X: %{x}<br>" +
            "Feature Y: %{y}<br>" +
            "Correlation: %{z:.2f}<extra></extra>"
        ),
        colorbar=dict(title="Correlation", titleside="right")
    ))

    # Annotate strong correlations
    threshold = 0.75
    annotations = []
    for i, feature_y in enumerate(features):
        for j, feature_x in enumerate(features):
            value = corr.iloc[i, j]
            if i != j and abs(value) >= threshold:
                annotations.append(dict(
                    x=feature_x,
                    y=feature_y,
                    text=f"{value:.2f}",
                    showarrow=False,
                    font=dict(color="black", size=16, family="Arial Black"),
                    bgcolor="rgba(255,255,255,0.7)"
                ))
    heatmap_fig.update_layout(
        title=dict(text=f"{method.capitalize()} Correlation Matrix", x=0.5, font=dict(size=28)),
        xaxis=dict(tickangle=-45, tickfont=dict(size=14)),
        yaxis=dict(tickfont=dict(size=14)),
        annotations=annotations,
        width=1100,
        height=900,
        paper_bgcolor='white',
        plot_bgcolor='rgba(0,0,0,0)'
    )
    heatmap_fig.show()
    
    # Optionally display a feature mapping table (only for VOLVE)
    if show_feature_mapping:
        
        if ds["name"].upper() == "VOLVE":
        
            default_feature_mapping = {
                "BORE_GAS_VOL": "Volume of gas produced in the well (SCF or m³).",
                "CE": "Energy Coefficient, a metric of production efficiency.",
                "AVG_DOWNHOLE_PRESSURE": "Average pressure at the bottom of the well (psi or bar).",
                "delta_P": "Pressure variation indicating operational changes.",
                "Tempo_Inicio_Prod": "Time since production start (hours/days).",
                "PI": "Well Productivity Index.",
                "AVG_WHP_P": "Average Wellhead Pressure.",
                "BORE_WAT_VOL": "Volume of water produced in the well.",
                "AVG_CHOKE_SIZE_P": "Average choke size controlling flow.",
                "BORE_WI_VOL_15_9_F_4": "Volume of water injection (SCF or m³).",
                "ON_STREAM_HRS": "Hours the well has been in production.",
                "Taxa_Declinio": "Rate of production decline."
            }
        elif ds["name"].upper() == "UNISIM":
            
            default_feature_mapping = {
                "PWFO": "Well pressure",
                "QOOB":"Oil flow",
                "QWOB":"Water flow",
                "QLOB":"Liquid flow (oil + water)",
                "QGOB":"Gas flow",
            }
            
        mapping_features = [feat for feat in features if feat in default_feature_mapping]
        mapping_descriptions = [default_feature_mapping.get(feat, "No description available.") for feat in mapping_features]
        table_fig = go.Figure(data=[go.Table(
            columnwidth=[0.25, 0.75],
            header=dict(
                values=["<b>Feature</b>", "<b>Description</b>"],
                fill_color=default_colors[0],
                font=dict(size=18, color='white'),
                align='left'
            ),
            cells=dict(
                values=[mapping_features , mapping_descriptions],
                fill_color=[['lavender'] * len(mapping_features),
                            ['#FFDDC1'] * len(mapping_descriptions)],
                align='left',
                font=dict(size=16, color=['black']),
                height=30
            )
        )])
        table_fig.update_layout(
            title=dict(text="Feature Mapping", x=0.5, font=dict(size=24)),
            width=1100,
            height=700,
            margin=dict(l=10, r=10, t=40, b=10),
            paper_bgcolor='white'
        )
        table_fig.show()


def perform_cross_correlation_analysis(df, feature_x, feature_y, max_lag=30, color_palette=None):
    """
    Compute and display an interactive cross-correlation stem plot between two features.
    """
    default_colors = ['#206A92', '#FF5733', '#8E44AD', '#E67E22', '#2ECC71']
    palette = color_palette if color_palette else default_colors
    line_color = palette[0]
    marker_color = palette[1] if len(palette) > 1 else palette[0]
    
    x = df[feature_x].dropna().values
    y = df[feature_y].dropna().values
    n = min(len(x), len(y))
    x, y = x[:n], y[:n]
    
    lags = np.arange(-max_lag, max_lag + 1)
    corr_values = []
    for lag in lags:
        if lag < 0:
            corr = np.corrcoef(x[:lag], y[-lag:])[0, 1]
        elif lag > 0:
            corr = np.corrcoef(x[lag:], y[:-lag])[0, 1]
        else:
            corr = np.corrcoef(x, y)[0, 1]
        corr_values.append(corr)
    
    fig = go.Figure()
    for lag, corr_val in zip(lags, corr_values):
        fig.add_shape(
            type="line",
            x0=lag, y0=0, x1=lag, y1=corr_val,
            line=dict(color=line_color, width=2)
        )
    fig.add_trace(go.Scatter(
        x=lags,
        y=corr_values,
        mode="markers",
        marker=dict(color=marker_color, size=10),
        name="Cross-Correlation"
    ))
    fig.add_shape(
        type="line",
        x0=min(lags), x1=max(lags),
        y0=0, y1=0,
        line=dict(color="gray", width=1, dash="dash")
    )
    fig.update_layout(
        title=dict(text=f"Cross-Correlation between {feature_x} and {feature_y}", x=0.5, font=dict(size=28)),
        xaxis=dict(title="Lag", tickfont=dict(size=14)),
        yaxis=dict(title="Cross-Correlation", tickfont=dict(size=14)),
        paper_bgcolor="white",
        plot_bgcolor="rgba(0,0,0,0)",
        width=900,
        height=600
    )
    fig.show()

# =============================================================================
# Main Pipeline
# =============================================================================
def main(selected_names=None):
    """
    Main function to load data and perform exploratory physical analysis.
    
    For each selected data source (e.g., VOLVE or UNISIM), load the data,
    generate correlation plots, cross-correlation plots, and run domain-specific tests.
    """
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

    # If no specific names are provided, process all data sources.
    if selected_names is None:
        selected_names = [ds["name"] for ds in DATA_SOURCES]
    filtered_data_sources = [ds for ds in DATA_SOURCES if ds["name"] in selected_names]
    
    # Default feature list for VOLVE
    volve_selected_features = [
        'BORE_OIL_VOL', 'BORE_GAS_VOL', 'CE', 'AVG_DOWNHOLE_PRESSURE', 'delta_P',
        'PI', 'AVG_WHP_P', 'BORE_WAT_VOL','AVG_CHOKE_SIZE_P', 'BORE_WI_VOL_15_9_F_4',
        'ON_STREAM_HRS', 'Taxa_Declinio', 'Tempo_Inicio_Prod'
    ]
    
    for ds in filtered_data_sources:
        # Get the appropriate feature list and main feature.
        features_list, main_feature = get_features_and_main(ds, volve_selected_features)
        
        for well in ds.get("wells", []):
        # for well in ['Prod-7']:
            logging.info(f"Processing well '{well}' from data source '{ds['name']}'")
            try:
                df = load_and_preprocess_data(DataSource, ds, features_list, well)
                logging.info(f"Data loaded for well '{well}' with shape: {df.shape}")
            except Exception as e:
                logging.error(f"Failed to load data for well '{well}': {e}")
                continue
            
            # Ensure we work only with available columns.
            features = [f for f in features_list if f in df.columns]
            if not features:
                logging.error(f"No matching features found for well '{well}'. Skipping analysis.")
                continue
            
            # 1. Correlation Analysis: Pearson and Spearman
            plot_correlation_matrix(df, features, ds, method="pearson")
            plot_correlation_matrix(df, features, ds, method="spearman")
            
            # 2. Cross-Correlation Analysis:
            # For demonstration, compare each feature against the main production metric.
            for feature in features:
                if feature in df.columns and main_feature in df.columns:
                    perform_cross_correlation_analysis(df, feature, main_feature, max_lag=20)
            
            # 3. Domain-Specific Consistency Checks:
            if ds["name"].upper() == "VOLVE":
                domain_specific_tests_volve(df, main_feature)
            elif ds["name"].upper() == "UNISIM":
                domain_specific_tests_unisim(df, main_feature)

if __name__ == "__main__":
    # For example, process only the VOLVE data source.
    main(selected_names=["VOLVE"])
    # To process UNISIM, call: main(selected_names=["UNISIM"])
